In [0]:
from datetime import timedelta
import pandas as pd
import numpy as np
import plotly.express as px

ACTUALS_PATH     = '/dbfs/mnt/thesis/output_data/processed_data.csv'
PREDICTIONS_PATH = '/dbfs/mnt/thesis/predictions/lgbm/ensemble_rolling_periodic_retune/'
ERRORS_PATH      = '/dbfs/mnt/thesis/mlops/lgbm/'

TODAY     = pd.to_datetime('2010-08-14')
YESTERDAY = TODAY - timedelta(days=1)

In [0]:
#Load data
actuals_df = pd.read_csv(ACTUALS_PATH, parse_dates=['DATETIME'])
preds_df   = pd.read_csv(
    PREDICTIONS_PATH + f"predictions_{YESTERDAY.date()}.csv",
    parse_dates=['DATETIME']
)

#Merge, filter zero‐actuals, compute errors 
eval_df = (
    preds_df
      .merge(actuals_df, on=['DATETIME','LOCATION'])
      .query("VALUE != 0")
      .rename(columns={'PREDICTED':'prediction','VALUE':'actual'})
)

eval_df['error']     = eval_df['prediction'] - eval_df['actual']
eval_df['abs_error'] = eval_df['error'].abs()
eval_df['sq_error']  = eval_df['error']**2
eval_df['DATE']      = eval_df['DATETIME'].dt.date

In [0]:
#Daily aggregation
def agg(group):
    mae          = group['abs_error'].mean()
    rmse         = np.sqrt(group['sq_error'].mean())
    mean_abs_act = group['actual'].abs().mean()
    nmae         = mae / mean_abs_act if mean_abs_act != 0 else np.nan
    return pd.Series({'MAE': mae, 'NMAE': nmae, 'RMSE': rmse})

daily_metrics = (
    eval_df
      .groupby(['DATE','LOCATION'])
      .apply(agg)
      .reset_index()
)
daily_metrics['DATE'] = pd.to_datetime(daily_metrics['DATE'])

#Append to rolling metrics history
METRICS_HISTORY_PATH = ERRORS_PATH + 'metrics_history.csv'
try:
    hist = pd.read_csv(METRICS_HISTORY_PATH, parse_dates=['DATE'])
except FileNotFoundError:
    hist = pd.DataFrame(columns=daily_metrics.columns)

hist = pd.concat([hist, daily_metrics], ignore_index=True)
hist = hist.drop_duplicates(['LOCATION','DATE'], keep='last')
cutoff = YESTERDAY - timedelta(days=14)
hist = hist[hist['DATE'] >= cutoff]
hist.to_csv(METRICS_HISTORY_PATH, index=False)

In [0]:
#Load retune history & reset baseline post-retune
RETUNE_LOG = ERRORS_PATH + 'retune_history.csv'
try:
    retune_log = pd.read_csv(RETUNE_LOG, parse_dates=['DATE'])
    last_rt = (
        retune_log
        .groupby('LOCATION')['DATE']
        .max()
        .rename('last_retune_date')
        .reset_index()
    )
except FileNotFoundError:
    last_rt = pd.DataFrame({
        'LOCATION': hist['LOCATION'].unique(),
        'last_retune_date': pd.to_datetime('1970-01-01')
    })

# join and filter out history up through last retune
hist2 = hist.merge(last_rt, on='LOCATION', how='left')
hist2['last_retune_date'] = hist2['last_retune_date'].fillna(pd.to_datetime('1970-01-01'))
hist2 = hist2[hist2['DATE'] > hist2['last_retune_date']]

#Compute rolling baseline_mean 
baseline = (
    hist2
      .sort_values(['LOCATION','DATE'])
      .groupby('LOCATION')
      .rolling(
          window=3,   
          on='DATE',
          min_periods=2,   
          closed='left' 
      )['NMAE']
      .mean()
      .reset_index()
      .rename(columns={'NMAE':'baseline_mean'})
)
baseline.to_csv(ERRORS_PATH + 'baseline_stats.csv', index=False)

In [0]:
#Load prior flags for persistence
FLAG_PATH = ERRORS_PATH + 'flag_status.csv'
try:
    prev_flags = pd.read_csv(FLAG_PATH, parse_dates=['DATE'])
except FileNotFoundError:
    prev_flags = pd.DataFrame(columns=[
        'LOCATION','DATE','rel_jump','abs_jump',
        'above_threshold','streak_count','flagged'
    ])

last_flags = (
    prev_flags
      .sort_values(['LOCATION','DATE'])
      .groupby('LOCATION')
      .tail(1)
      .set_index('LOCATION')
      [['above_threshold','streak_count']]
      .rename(columns={
          'above_threshold':'above_yesterday',
          'streak_count'    :'streak_yesterday'
      })
)

#Flag logic
status = daily_metrics.merge(baseline, on=['LOCATION','DATE'], how='left')

#thresholds
REL_JUMP  = 1.10
ABS_DELTA = 0.02

#individual criteria
status['rel_jump'] = status['NMAE'] > REL_JUMP * status['baseline_mean']
status['abs_jump'] = (status['NMAE'] - status['baseline_mean']) > ABS_DELTA

#combined today
status['above_threshold'] = (status['rel_jump'] | status['abs_jump']).fillna(False)

#bring in yesterday’s state
status = status.merge(
    last_flags,
    how='left',
    left_on='LOCATION',
    right_index=True
)

status['above_yesterday']  = status['above_yesterday'].fillna(False)
status['streak_yesterday'] = status['streak_yesterday'].fillna(0).astype(int)

#compute new streak
status['streak_count'] = np.where(
    status['above_threshold'],
    status['streak_yesterday'] + 1,
    0
)

status['flagged'] = status['streak_count'] >= 2

In [0]:
new_flags = status[[
    'LOCATION','DATE','rel_jump','abs_jump',
    'above_threshold','streak_count','flagged'
]]
all_flags = pd.concat([prev_flags, new_flags], ignore_index=True)
all_flags = all_flags.drop_duplicates(['LOCATION','DATE'], keep='last')
all_flags.to_csv(FLAG_PATH, index=False)

#Log
RETUNE_LOG = ERRORS_PATH + 'retune_history.csv'
today_retunes = status.loc[
    status['flagged'],
    ['LOCATION','DATE','NMAE','rel_jump','abs_jump']
].copy()
today_retunes['retuned'] = True

try:
    log = pd.read_csv(RETUNE_LOG, parse_dates=['DATE'])
except FileNotFoundError:
    log = pd.DataFrame(columns=today_retunes.columns)

log = pd.concat([log, today_retunes], ignore_index=True)
log = log.drop_duplicates(['LOCATION','DATE'], keep='last')
log.to_csv(RETUNE_LOG, index=False)

In [0]:
locs_to_retune = status.loc[status['flagged'], 'LOCATION'].unique().tolist()

print("Retune these locations:", locs_to_retune)

Retune these locations: []


- 05: Retune these locations: [19, 21, 31, 47]
- 06: -
- 07: Retune these locations: [2, 7, 22, 23, 30, 38, 39, 40, 46]
- 08: Retune these locations: [10, 29, 45]
- 09: Retune these locations: [0, 9, 11, 19, 21, 25, 35, 41, 44]
- 10: Retune these locations: [14, 15]
- 11: Retune these locations: [20, 33, 37, 38, 39]
- 12: Retune these locations: [17]
- 13: Retune these locations: [6, 7, 21, 27, 28, 30]
- 14: Retune these locations: []